# Multi-model TAVI inference: SwinUNETR + SegResNet

Questo notebook esegue inferenza su tutti i pazienti `TAVI_*` in `notebooks/tavi_to_infer/`, usando:

- `SwinUNETR/best_metric_model.pth`
- `SwinUNETR/latest_checkpoint.pth`
- `SegResNet/best_metric_model.pth`
- `SegResNet/latest_checkpoint.pth`

Per ogni paziente cerca 4 versioni del volume CT:

- `CT_LATE_*_denoising_contrasto.nii.gz`
- `CT_LATE_*_original.nii.gz`
- `CT_LATE_*_solo_contrasto.nii.gz`
- `CT_LATE_*_solo_denoising.nii.gz`

Produce segmentazioni `.nii.gz` e un CSV per ogni paziente con Dice, surface Hausdorff, Hausdorff, IoU e volumi. Se manca la ground truth, salva comunque la segmentazione e calcola solo il volume predetto, perché anche le metriche, poverine, hanno bisogno di qualcosa con cui confrontarsi.

La ground truth viene letta da `notebooks/tavi_to_infer/TAVI_**/registration_mask.nii.gz`.


In [1]:
import os
import sys
from pathlib import Path

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/mpl-cache")
os.environ.setdefault("XDG_CACHE_HOME", "/private/tmp/xdg-cache")

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.executable}")

Project root: /Users/ricca/Desktop/Health_Informatics_Internship_2026
Python: /opt/miniconda3/bin/python


In [2]:
from dataclasses import dataclass
import csv
import inspect
import math
import shutil
import traceback
from contextlib import nullcontext

import nibabel as nib
import numpy as np
import pandas as pd
from scipy.ndimage import binary_erosion, generate_binary_structure
from scipy.spatial import cKDTree
from omegaconf import OmegaConf

import torch
from monai.data import DataLoader, Dataset, MetaTensor, decollate_batch
from monai.inferers import sliding_window_inference
from monai.transforms import AsDiscreted, Compose, Invertd

from lems_ct.src.utils.transforms import get_transforms
from scripts.lcc_postprocessing import keep_largest_cc_numpy

print(f"torch: {torch.__version__}")
print(f"MPS built: {torch.backends.mps.is_built()}")
print(f"MPS available: {torch.backends.mps.is_available()}")

torch: 2.11.0
MPS built: True
MPS available: True


In [3]:
# ===== Configurazione principale =====

INPUT_ROOT = Path("notebooks/tavi_to_infer")
OUTPUT_ROOT = Path("notebooks/output/tavi_to_infer_segmentations")
CONFIG_PATH = Path("config/train_config.yaml")

DEVICE = "mps"  # "mps", "cuda", "cpu" oppure "auto"
NUM_WORKERS = 0
SKIP_EXISTING = True
RUN_LCC_POSTPROCESSING = True  # True: salva segmentation_model.nii.gz dopo Largest Connected Component
SANITIZE_NIFTI_FOR_MONAI = True  # crea copie NIfTI pulite per evitare crash di MONAI LoadImaged
MONAI_CACHE_ROOT = OUTPUT_ROOT / "_monai_readable_inputs"

# Se vuoi testare su pochi pazienti prima di lanciare tutto, metti un numero intero.
MAX_PATIENTS = None

CHECKPOINTS = {
    "SwinUNETR": {
        "best_metric_model": Path("notebooks/output/models/SwinUNETR/best_metric_model.pth"),
        "latest_checkpoint": Path("notebooks/output/models/SwinUNETR/latest_checkpoint.pth"),
    },
    "SegResNet": {
        "best_metric_model": Path("notebooks/output/models/SegResNet/best_metric_model.pth"),
        "latest_checkpoint": Path("notebooks/output/models/SegResNet/latest_checkpoint.pth"),
    },
}

# Pattern volutamente generali: funzionano per TAVI_100 ma anche per TAVI_002, ecc.
VOLUME_VARIANTS = {
    "denoising_contrasto": "CT_LATE_*_denoising_contrasto.nii.gz",
    "original": "CT_LATE_*_original.nii.gz",
    "solo_contrasto": "CT_LATE_*_solo_contrasto.nii.gz",
    "solo_denoising": "CT_LATE_*_solo_denoising.nii.gz",
}

# Ground truth ufficiale per ogni TAVI.
LABEL_CANDIDATES = ["registration_mask.nii.gz"]

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
MONAI_CACHE_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Input root exists: {INPUT_ROOT.exists()} | {INPUT_ROOT.resolve()}")
print(f"Output root: {OUTPUT_ROOT.resolve()}")
print(f"Config exists: {CONFIG_PATH.exists()} | {CONFIG_PATH}")

Input root exists: True | /Users/ricca/Desktop/Health_Informatics_Internship_2026/notebooks/tavi_to_infer
Output root: /Users/ricca/Desktop/Health_Informatics_Internship_2026/notebooks/output/tavi_to_infer_segmentations
Config exists: True | config/train_config.yaml


In [4]:
def resolve_project_path(path: str | Path) -> Path:
    path = Path(path)
    return path if path.is_absolute() else PROJECT_ROOT / path


def select_device(requested: str) -> torch.device:
    requested = requested.lower()
    mps_available = bool(torch.backends.mps.is_available())
    if requested == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA richiesta, ma torch.cuda.is_available() è False.")
    if requested == "mps" and not mps_available:
        raise RuntimeError("MPS richiesto, ma torch.backends.mps.is_available() è False. Usa DEVICE='cpu' o DEVICE='auto'.")
    if requested == "cuda":
        return torch.device("cuda")
    if requested == "mps":
        return torch.device("mps")
    if requested == "cpu":
        return torch.device("cpu")
    if torch.cuda.is_available():
        return torch.device("cuda")
    if mps_available:
        return torch.device("mps")
    return torch.device("cpu")


def as_binary_mask(mask: np.ndarray) -> np.ndarray:
    mask = np.asarray(mask)
    if mask.ndim == 4 and 1 in (mask.shape[0], mask.shape[-1]):
        mask = np.squeeze(mask)
    if mask.ndim != 3:
        raise ValueError(f"Expected a 3D mask, received shape {mask.shape}.")
    return mask > 0


def surface_voxels(mask: np.ndarray) -> np.ndarray:
    foreground = as_binary_mask(mask)
    if not foreground.any():
        return foreground
    structure = generate_binary_structure(rank=3, connectivity=1)
    eroded = binary_erosion(foreground, structure=structure, border_value=0)
    return foreground & ~eroded


def directed_distances(source: np.ndarray, target: np.ndarray, spacing: tuple[float, float, float]) -> np.ndarray:
    source_points = np.argwhere(source) * np.asarray(spacing, dtype=float)
    target_points = np.argwhere(target) * np.asarray(spacing, dtype=float)
    if source_points.size == 0 or target_points.size == 0:
        return np.asarray([], dtype=float)
    return cKDTree(target_points).query(source_points, k=1)[0]


def symmetric_distance_stats(first: np.ndarray, second: np.ndarray, spacing: tuple[float, float, float]) -> tuple[float, float]:
    first = as_binary_mask(first)
    second = as_binary_mask(second)
    if not first.any() and not second.any():
        return 0.0, 0.0
    if not first.any() or not second.any():
        return math.inf, math.inf
    distances = np.concatenate([
        directed_distances(first, second, spacing),
        directed_distances(second, first, spacing),
    ])
    return float(np.max(distances)), float(np.percentile(distances, 95))


def nifti_spacing(path: str | Path) -> tuple[float, float, float]:
    image = nib.load(str(path))
    return tuple(float(value) for value in image.header.get_zooms()[:3])


def load_binary_nifti(path: str | Path) -> np.ndarray:
    return as_binary_mask(nib.load(str(path)).get_fdata()).astype(np.uint8)

def sanitize_nifti_for_monai(src_path: str | Path, dst_path: str | Path, is_label: bool) -> Path:
    """Rewrite a NIfTI with a simple dtype/header so MONAI LoadImaged can read it reliably.

    This keeps affine/qform/sform from the original file but removes the usual NIfTI
    header weirdness that can make LoadImaged explode with a useless generic error.
    """
    src_path = resolve_project_path(src_path)
    dst_path = resolve_project_path(dst_path)
    dst_path.parent.mkdir(parents=True, exist_ok=True)

    # Reuse cache only when it is newer than the source. Human civilization advances slowly,
    # but not so slowly that we should rewrite gigabytes every run.
    if dst_path.exists() and dst_path.stat().st_mtime >= src_path.stat().st_mtime:
        return dst_path

    img = nib.load(str(src_path))
    if is_label:
        data = np.asanyarray(img.dataobj)
        data = (data > 0).astype(np.uint8)
    else:
        data = img.get_fdata(dtype=np.float32).astype(np.float32)
        data = np.nan_to_num(data, nan=0.0, posinf=0.0, neginf=0.0)

    header = img.header.copy()
    header.set_data_dtype(np.uint8 if is_label else np.float32)
    clean = nib.Nifti1Image(data, affine=img.affine, header=header)

    qform_code = int(np.asarray(img.header["qform_code"]).item())
    sform_code = int(np.asarray(img.header["sform_code"]).item())
    clean.set_qform(img.get_qform(), code=qform_code)
    clean.set_sform(img.get_sform(), code=sform_code)
    nib.save(clean, str(dst_path))
    return dst_path


def monai_cache_paths(volume) -> tuple[Path, Path]:
    image_cache = MONAI_CACHE_ROOT / volume.patient_id / volume.variant / "image_float32.nii.gz"
    if volume.label_path is not None:
        label_cache = MONAI_CACHE_ROOT / volume.patient_id / "registration_mask_uint8.nii.gz"
    else:
        label_cache = MONAI_CACHE_ROOT / volume.patient_id / volume.variant / "dummy_label_from_image_uint8.nii.gz"
    return image_cache, label_cache


def validate_nifti_readable(path: str | Path) -> tuple[bool, str | None]:
    try:
        img = nib.load(str(resolve_project_path(path)))
        _ = img.shape
        _ = img.header.get_zooms()
        return True, None
    except Exception as exc:
        return False, f"{type(exc).__name__}: {exc}"


def volume_ml(mask: np.ndarray, spacing: tuple[float, float, float]) -> float:
    voxel_volume_ml = float(np.prod(spacing)) / 1000.0
    return float(as_binary_mask(mask).sum() * voxel_volume_ml)


def binary_mask_metrics(pred_mask: np.ndarray, ground_truth_mask: np.ndarray | None, spacing: tuple[float, float, float]) -> dict[str, float | int | None]:
    pred = as_binary_mask(pred_mask)
    pred_voxels = int(pred.sum())
    pred_volume_ml = volume_ml(pred, spacing)

    if ground_truth_mask is None:
        return {
            "dice": None,
            "surface_hausdorff_mm": None,
            "surface_hausdorff95_mm": None,
            "hausdorff_mm": None,
            "hausdorff95_mm": None,
            "iou": None,
            "pred_volume_ml": pred_volume_ml,
            "ground_truth_volume_ml": None,
            "volume_difference_ml": None,
            "volume_ratio_pred_to_gt": None,
            "pred_voxels": pred_voxels,
            "ground_truth_voxels": None,
        }

    gt = as_binary_mask(ground_truth_mask)
    gt_voxels = int(gt.sum())
    intersection = int(np.logical_and(pred, gt).sum())
    union = int(np.logical_or(pred, gt).sum())

    dice = 1.0 if pred_voxels + gt_voxels == 0 else 2.0 * intersection / float(pred_voxels + gt_voxels)
    iou = 1.0 if union == 0 else intersection / float(union)
    surface_hd, surface_hd95 = symmetric_distance_stats(surface_voxels(pred), surface_voxels(gt), spacing)
    full_hd, full_hd95 = symmetric_distance_stats(pred, gt, spacing)
    gt_volume_ml = volume_ml(gt, spacing)

    return {
        "dice": float(dice),
        "surface_hausdorff_mm": surface_hd,
        "surface_hausdorff95_mm": surface_hd95,
        "hausdorff_mm": full_hd,
        "hausdorff95_mm": full_hd95,
        "iou": float(iou),
        "pred_volume_ml": float(pred_volume_ml),
        "ground_truth_volume_ml": float(gt_volume_ml),
        "volume_difference_ml": float(pred_volume_ml - gt_volume_ml),
        "volume_ratio_pred_to_gt": math.inf if gt_volume_ml == 0 else float(pred_volume_ml / gt_volume_ml),
        "pred_voxels": pred_voxels,
        "ground_truth_voxels": gt_voxels,
    }


def save_mask_like_reference(mask: np.ndarray, reference_path: str | Path, output_path: str | Path) -> None:
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    reference = nib.load(str(reference_path))
    data = as_binary_mask(mask).astype(np.uint8)
    image = nib.Nifti1Image(data, affine=reference.affine, header=reference.header.copy())
    image.set_data_dtype(np.uint8)
    qform_code = int(np.asarray(reference.header["qform_code"]).item())
    sform_code = int(np.asarray(reference.header["sform_code"]).item())
    image.set_qform(reference.get_qform(), code=qform_code)
    image.set_sform(reference.get_sform(), code=sform_code)
    nib.save(image, str(output_path))


def write_csv(path: str | Path, rows: list[dict], columns: list[str] | None = None) -> None:
    """Write rows to CSV safely, even when rows is empty.

    `columns` is useful for error logs: pandas cannot read an empty, headerless CSV
    later, because apparently the void is not a valid table.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if rows:
        df = pd.DataFrame(rows)
        if columns is not None:
            for col in columns:
                if col not in df.columns:
                    df[col] = None
            df = df[columns + [col for col in df.columns if col not in columns]]
    else:
        df = pd.DataFrame(columns=columns or [])
    df.to_csv(path, index=False)


def load_checkpoint_state(checkpoint_path: Path, use_ema_weights: bool = True) -> dict:
    # Compatibilità con checkpoint salvati con versioni diverse di numpy/torch. Il teatrino, purtroppo, serve.
    if not hasattr(np, "_core") and hasattr(np, "core"):
        sys.modules.setdefault("numpy._core", np.core)

    try:
        checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    except TypeError:
        checkpoint = torch.load(checkpoint_path, map_location="cpu")

    if isinstance(checkpoint, dict):
        if use_ema_weights and checkpoint.get("ema_model_state_dict") is not None:
            state_dict = checkpoint["ema_model_state_dict"]
            print(f"Loaded ema_model_state_dict from {checkpoint_path.name}")
        elif "model_state_dict" in checkpoint:
            state_dict = checkpoint["model_state_dict"]
            print(f"Loaded model_state_dict from {checkpoint_path.name}")
        elif "state_dict" in checkpoint:
            state_dict = checkpoint["state_dict"]
            print(f"Loaded state_dict from {checkpoint_path.name}")
        else:
            state_dict = checkpoint
            print(f"Loaded raw checkpoint dict from {checkpoint_path.name}")
    else:
        state_dict = checkpoint

    if not isinstance(state_dict, dict):
        raise TypeError(f"Unsupported checkpoint format in {checkpoint_path}")

    cleaned = {}
    for key, value in state_dict.items():
        key = key[7:] if key.startswith("module.") else key
        key = key[6:] if key.startswith("model.") else key
        cleaned[key] = value
    return cleaned


def as_3d_numpy_mask(monai_pred) -> np.ndarray:
    if hasattr(monai_pred, "detach"):
        array = monai_pred.detach().cpu().numpy()
    else:
        array = np.asarray(monai_pred)
    return as_binary_mask(array).astype(np.uint8)

In [5]:
@dataclass(frozen=True)
class InferenceVolume:
    patient_id: str
    patient_dir: Path
    variant: str
    image_path: Path
    label_path: Path | None


def find_label_path(patient_dir: Path) -> Path | None:
    # La GT corretta è notebooks/tavi_to_infer/TAVI_**/registration_mask.nii.gz
    candidate = patient_dir / "registration_mask.nii.gz"
    return candidate if candidate.exists() else None


def find_variant_volume(patient_dir: Path, pattern: str) -> Path | None:
    matches = sorted(patient_dir.glob(pattern))
    if not matches:
        return None
    if len(matches) > 1:
        print(f"[{patient_dir.name}] più file matchano {pattern}; uso il primo: {matches[0].name}")
    return matches[0]


def discover_inference_volumes(input_root: Path) -> tuple[list[InferenceVolume], list[dict]]:
    volumes: list[InferenceVolume] = []
    missing_rows: list[dict] = []
    patient_dirs = sorted(path for path in input_root.glob("TAVI_*") if path.is_dir())
    if MAX_PATIENTS is not None:
        patient_dirs = patient_dirs[: int(MAX_PATIENTS)]

    for patient_dir in patient_dirs:
        label_path = find_label_path(patient_dir)
        for variant, pattern in VOLUME_VARIANTS.items():
            image_path = find_variant_volume(patient_dir, pattern)
            if image_path is None:
                missing_rows.append({
                    "patient_id": patient_dir.name,
                    "variant": variant,
                    "expected_pattern": pattern,
                    "problem": "missing_image",
                })
                continue
            volumes.append(InferenceVolume(
                patient_id=patient_dir.name,
                patient_dir=patient_dir,
                variant=variant,
                image_path=image_path,
                label_path=label_path,
            ))
        if label_path is None:
            missing_rows.append({
                "patient_id": patient_dir.name,
                "variant": "ALL",
                "expected_pattern": "; ".join(LABEL_CANDIDATES),
                "problem": "missing_ground_truth_metrics_will_be_partial",
            })
    return volumes, missing_rows


volumes, discovery_warnings = discover_inference_volumes(INPUT_ROOT)
print(f"Pazienti trovati: {len(set(v.patient_id for v in volumes))}")
print(f"Volumi da inferire: {len(volumes)}")
print(f"Warning discovery: {len(discovery_warnings)}")

preview = pd.DataFrame([
    {
        "patient_id": v.patient_id,
        "variant": v.variant,
        "image": str(v.image_path),
        "label": str(v.label_path) if v.label_path else None,
    }
    for v in volumes
])
display(preview.head(20))

if discovery_warnings:
    warnings_df = pd.DataFrame(discovery_warnings)
    write_csv(OUTPUT_ROOT / "discovery_warnings.csv", discovery_warnings)
    display(warnings_df.head(30))
# Preflight: verifica lettura con nibabel prima di entrare nel circo di MONAI.
readability_rows = []
for v in volumes:
    image_ok, image_err = validate_nifti_readable(v.image_path)
    label_ok, label_err = (True, None)
    if v.label_path is not None:
        label_ok, label_err = validate_nifti_readable(v.label_path)
    readability_rows.append({
        "patient_id": v.patient_id,
        "variant": v.variant,
        "image_readable": image_ok,
        "image_error": image_err,
        "label_readable": label_ok,
        "label_error": label_err,
        "label_path": str(v.label_path) if v.label_path else None,
    })

readability_df = pd.DataFrame(readability_rows)
display(readability_df.head(20))
write_csv(OUTPUT_ROOT / "nifti_readability_preflight.csv", readability_rows)

bad = readability_df[(readability_df["image_readable"] == False) | (readability_df["label_readable"] == False)]
if len(bad):
    display(bad)
    raise RuntimeError(f"Ci sono {len(bad)} file NIfTI non leggibili da nibabel. Dettagli in nifti_readability_preflight.csv")


Pazienti trovati: 2
Volumi da inferire: 8
Warning discovery: 0


,patient_id,variant,image,label
0,TAVI_100,denoising_contrasto,notebooks/tavi_to_infer/TAVI_100/CT_LATE_100_d...,notebooks/tavi_to_infer/TAVI_100/registration_...
1,TAVI_100,original,notebooks/tavi_to_infer/TAVI_100/CT_LATE_100_o...,notebooks/tavi_to_infer/TAVI_100/registration_...
2,TAVI_100,solo_contrasto,notebooks/tavi_to_infer/TAVI_100/CT_LATE_100_s...,notebooks/tavi_to_infer/TAVI_100/registration_...
3,TAVI_100,solo_denoising,notebooks/tavi_to_infer/TAVI_100/CT_LATE_100_s...,notebooks/tavi_to_infer/TAVI_100/registration_...
4,TAVI_35,denoising_contrasto,notebooks/tavi_to_infer/TAVI_35/CT_LATE_35_den...,notebooks/tavi_to_infer/TAVI_35/registration_m...
5,TAVI_35,original,notebooks/tavi_to_infer/TAVI_35/CT_LATE_35_ori...,notebooks/tavi_to_infer/TAVI_35/registration_m...
6,TAVI_35,solo_contrasto,notebooks/tavi_to_infer/TAVI_35/CT_LATE_35_sol...,notebooks/tavi_to_infer/TAVI_35/registration_m...
7,TAVI_35,solo_denoising,notebooks/tavi_to_infer/TAVI_35/CT_LATE_35_sol...,notebooks/tavi_to_infer/TAVI_35/registration_m...


,patient_id,variant,image_readable,image_error,label_readable,label_error,label_path
0,TAVI_100,denoising_contrasto,True,None,True,None,notebooks/tavi_to_infer/TAVI_100/registration_...
1,TAVI_100,original,True,None,True,None,notebooks/tavi_to_infer/TAVI_100/registration_...
2,TAVI_100,solo_contrasto,True,None,True,None,notebooks/tavi_to_infer/TAVI_100/registration_...
3,TAVI_100,solo_denoising,True,None,True,None,notebooks/tavi_to_infer/TAVI_100/registration_...
4,TAVI_35,denoising_contrasto,True,None,True,None,notebooks/tavi_to_infer/TAVI_35/registration_m...
5,TAVI_35,original,True,None,True,None,notebooks/tavi_to_infer/TAVI_35/registration_m...
6,TAVI_35,solo_contrasto,True,None,True,None,notebooks/tavi_to_infer/TAVI_35/registration_m...
7,TAVI_35,solo_denoising,True,None,True,None,notebooks/tavi_to_infer/TAVI_35/registration_m...


In [6]:
def kwargs_for_signature(raw_cfg: dict, signature_keys) -> dict:
    return {key: value for key, value in raw_cfg.items() if key in signature_keys}


def build_swin_unetr_model(cfg):
    from monai.networks.nets import SwinUNETR
    try:
        from scripts.swin_UNETR import swin_kwargs_for_signature, with_swin_defaults
        signature_keys = inspect.signature(SwinUNETR).parameters.keys()
        model_cfg = OmegaConf.to_container(cfg.get("model", {}), resolve=True) or {}
        roi_size = OmegaConf.to_container(cfg.transforms.roi_size, resolve=True)
        swin_cfg = with_swin_defaults(model_cfg, roi_size)
        kwargs = swin_kwargs_for_signature(swin_cfg, signature_keys)
        print("SwinUNETR kwargs:", kwargs)
        return SwinUNETR(**kwargs)
    except Exception as exc:
        print(f"Fallback SwinUNETR builder usato perché scripts.swin_UNETR non è importabile o non compatibile: {exc}")
        signature_keys = inspect.signature(SwinUNETR).parameters.keys()
        model_cfg = OmegaConf.to_container(cfg.get("model", {}), resolve=True) or {}
        roi_size = tuple(int(v) for v in cfg.transforms.roi_size)
        defaults = {
            "img_size": roi_size,       # usato da MONAI vecchie versioni
            "spatial_dims": 3,
            "in_channels": 1,
            "out_channels": 2,
            "feature_size": 48,
            "use_checkpoint": True,
        }
        defaults.update(model_cfg)
        kwargs = kwargs_for_signature(defaults, signature_keys)
        print("SwinUNETR fallback kwargs:", kwargs)
        return SwinUNETR(**kwargs)


def build_segresnet_model(cfg):
    from monai.networks.nets import SegResNet
    signature_keys = inspect.signature(SegResNet).parameters.keys()
    model_cfg = OmegaConf.to_container(cfg.get("model", {}), resolve=True) or {}

    # Se il tuo config ha una sezione specifica per SegResNet, usa quella.
    for key in ["segresnet", "SegResNet", "seg_res_net"]:
        if key in model_cfg and isinstance(model_cfg[key], dict):
            model_cfg = model_cfg[key]
            break

    defaults = {
        "spatial_dims": 3,
        "in_channels": 1,
        "out_channels": 2,
        "init_filters": 32,
        "blocks_down": (1, 2, 2, 4),
        "blocks_up": (1, 1, 1),
        "dropout_prob": 0.0,
    }
    defaults.update(model_cfg)

    # Rimuove chiavi tipiche di SwinUNETR se il config è condiviso. Sì, i config multi-modello sono un parco giochi minato.
    for bad_key in [
        "img_size", "roi_size", "feature_size", "depths", "num_heads", "window_size",
        "mlp_ratio", "qkv_bias", "drop_rate", "attn_drop_rate", "dropout_path_rate",
        "use_checkpoint", "downsample", "use_v2", "norm_name",
    ]:
        if bad_key not in signature_keys:
            defaults.pop(bad_key, None)

    kwargs = kwargs_for_signature(defaults, signature_keys)
    print("SegResNet kwargs:", kwargs)
    return SegResNet(**kwargs)


MODEL_BUILDERS = {
    "SwinUNETR": build_swin_unetr_model,
    "SegResNet": build_segresnet_model,
}


def load_model(model_name: str, checkpoint_path: Path, cfg, device: torch.device) -> torch.nn.Module:
    checkpoint_path = resolve_project_path(checkpoint_path)
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Checkpoint non trovato: {checkpoint_path}")
    model = MODEL_BUILDERS[model_name](cfg).to(device)
    state_dict = load_checkpoint_state(checkpoint_path, use_ema_weights=True)
    incompatible = model.load_state_dict(state_dict, strict=False)
    if incompatible.missing_keys:
        print(f"[{model_name}] missing keys: {len(incompatible.missing_keys)}")
        print(incompatible.missing_keys[:10])
    if incompatible.unexpected_keys:
        print(f"[{model_name}] unexpected keys: {len(incompatible.unexpected_keys)}")
        print(incompatible.unexpected_keys[:10])
    model.eval()
    return model


cfg = OmegaConf.load(resolve_project_path(CONFIG_PATH))
device = select_device(DEVICE)
amp_context = torch.amp.autocast("cuda") if device.type == "cuda" else nullcontext()
roi_size = tuple(int(value) for value in cfg.transforms.roi_size)
sw_batch_size = int(cfg.inference.get("sw_batch_size", 4))
overlap = float(cfg.inference.get("overlap", 0.5))
mode = str(cfg.inference.get("mode", "gaussian"))

_, val_transforms = get_transforms(**cfg.transforms)
post_transforms = Compose([
    Invertd(
        keys="pred",
        transform=val_transforms,
        orig_keys="image",
        meta_keys="pred_meta_dict",
        orig_meta_keys="image_meta_dict",
        meta_key_postfix="meta_dict",
        nearest_interp=False,
        to_tensor=True,
    ),
    AsDiscreted(keys="pred", argmax=True),
])

print(f"Device: {device}")
print(f"roi_size={roi_size}, sw_batch_size={sw_batch_size}, overlap={overlap}, mode={mode}")

Device: mps
roi_size=(96, 96, 96), sw_batch_size=4, overlap=0.5, mode=gaussian


/opt/miniconda3/lib/python3.13/site-packages/monai/utils/deprecate_utils.py:320: FutureWarning: monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
  warn_deprecated(argname, msg, warning_category)


In [7]:
def output_paths(volume: InferenceVolume, model_name: str, checkpoint_name: str) -> dict[str, Path]:
    base_dir = OUTPUT_ROOT / volume.patient_id / volume.variant / model_name / checkpoint_name
    return {
        "base_dir": base_dir,
        "mask_path": base_dir / "segmentation_model.nii.gz",
        "raw_mask_path": base_dir / "segmentation_model_raw.nii.gz",
        "input_copy_path": base_dir / volume.image_path.name,
        "gt_copy_path": base_dir / "ground_truth_mask.nii.gz",
    }


def make_monai_case(volume: InferenceVolume) -> dict:
    """Create a MONAI dict compatible with the validation transforms.

    Fix principale: i path passati a LoadImaged sono copie NIfTI pulite, assolute,
    create via nibabel. La GT reale rimane `registration_mask.nii.gz` e viene usata
    per le metriche; la copia in cache serve solo alla pipeline MONAI.
    """
    image_path_abs = resolve_project_path(volume.image_path)
    label_path_abs = resolve_project_path(volume.label_path) if volume.label_path is not None else None

    if SANITIZE_NIFTI_FOR_MONAI:
        image_cache, label_cache = monai_cache_paths(volume)
        image_for_monai = sanitize_nifti_for_monai(image_path_abs, image_cache, is_label=False)
        if label_path_abs is not None:
            label_for_monai = sanitize_nifti_for_monai(label_path_abs, label_cache, is_label=True)
        else:
            # Dummy label solo per compatibilità con transform validation che richiedono `label`.
            label_for_monai = sanitize_nifti_for_monai(image_path_abs, label_cache, is_label=True)
    else:
        image_for_monai = image_path_abs
        label_for_monai = label_path_abs if label_path_abs is not None else image_path_abs

    return {
        "image": str(image_for_monai.resolve()),
        "label": str(label_for_monai.resolve()),
    }


def run_single_volume_inference(
    model: torch.nn.Module,
    volume: InferenceVolume,
    model_name: str,
    checkpoint_name: str,
) -> dict:
    paths = output_paths(volume, model_name, checkpoint_name)
    paths["base_dir"].mkdir(parents=True, exist_ok=True)
    mask_path = paths["mask_path"]

    image_path_abs = resolve_project_path(volume.image_path)
    label_path_abs = resolve_project_path(volume.label_path) if volume.label_path is not None else None

    if not image_path_abs.exists():
        raise FileNotFoundError(f"Volume immagine non trovato: {image_path_abs}")
    if label_path_abs is not None and not label_path_abs.exists():
        raise FileNotFoundError(f"Ground truth non trovata: {label_path_abs}")

    # `mask_path` è sempre la maschera finale post-processata con Largest Connected Component.
    # `raw_mask_path` è la predizione binaria prima del LCC, utile per debug/confronto.
    can_skip = SKIP_EXISTING and mask_path.exists() and (not RUN_LCC_POSTPROCESSING or paths["raw_mask_path"].exists())
    if can_skip:
        pred_mask = load_binary_nifti(mask_path)
        status = "skipped_existing"
    else:
        # val_transforms arriva dal training e di solito si aspetta anche `label`.
        # make_monai_case prepara path assoluti e NIfTI puliti per LoadImaged.
        data = [make_monai_case(volume)]

        dataset = Dataset(data=data, transform=val_transforms)
        loader = DataLoader(dataset, batch_size=1, num_workers=NUM_WORKERS)
        batch = next(iter(loader))
        inputs = batch["image"].to(device)

        with torch.inference_mode():
            with amp_context:
                logits = sliding_window_inference(
                    inputs,
                    roi_size,
                    sw_batch_size,
                    model,
                    overlap=overlap,
                    mode=mode,
                )
                probs = torch.softmax(logits, dim=1)

        src_meta = batch["image"].meta if hasattr(batch["image"], "meta") else batch.get("image_meta_dict")
        pred_tensor = MetaTensor(probs.detach().cpu(), meta=src_meta)
        item = decollate_batch({"image": batch["image"], "pred": pred_tensor})[0]
        if "image_meta_dict" not in item:
            item["image_meta_dict"] = item["image"].meta if hasattr(item["image"], "meta") else src_meta
        if "pred_meta_dict" not in item:
            item["pred_meta_dict"] = dict(item["image_meta_dict"])
        item = post_transforms(item)

        raw_mask = as_3d_numpy_mask(item["pred"])
        pred_mask = keep_largest_cc_numpy(raw_mask) if RUN_LCC_POSTPROCESSING else raw_mask

        # Salvataggio esplicito:
        # - segmentation_model_raw.nii.gz = maschera predetta prima del LCC
        # - segmentation_model.nii.gz = maschera finale, con LCC se RUN_LCC_POSTPROCESSING=True
        save_mask_like_reference(raw_mask, image_path_abs, paths["raw_mask_path"])
        save_mask_like_reference(pred_mask, image_path_abs, mask_path)
        shutil.copy2(image_path_abs, paths["input_copy_path"])
        if label_path_abs is not None:
            shutil.copy2(label_path_abs, paths["gt_copy_path"])
        status = "computed"

    spacing = nifti_spacing(image_path_abs)
    gt_mask = load_binary_nifti(label_path_abs) if label_path_abs is not None else None
    metrics = binary_mask_metrics(pred_mask, gt_mask, spacing)

    row = {
        "patient_id": volume.patient_id,
        "variant": volume.variant,
        "model": model_name,
        "checkpoint": checkpoint_name,
        "status": status,
        "image_path": str(image_path_abs),
        "label_path": str(label_path_abs) if label_path_abs else None,
        "mask_path": str(mask_path),
        "raw_mask_path": str(paths["raw_mask_path"]),
        "spacing_x_mm": spacing[0],
        "spacing_y_mm": spacing[1],
        "spacing_z_mm": spacing[2],
        "lcc_postprocessing": RUN_LCC_POSTPROCESSING,
        "has_ground_truth": label_path_abs is not None,
    }
    row.update(metrics)
    return row


def write_patient_csvs(rows: list[dict]) -> None:
    all_df = pd.DataFrame(rows)
    write_csv(OUTPUT_ROOT / "all_tavi_metrics.csv", rows)

    if all_df.empty:
        print("Nessuna riga metrica da salvare per paziente. Controlla inference_errors.csv, il solito diario delle sciagure.")
        return

    for patient_id, patient_df in all_df.groupby("patient_id"):
        patient_dir = OUTPUT_ROOT / patient_id
        patient_dir.mkdir(parents=True, exist_ok=True)
        patient_df = patient_df.sort_values(["variant", "model", "checkpoint"])
        patient_df.to_csv(patient_dir / f"{patient_id}_metrics.csv", index=False)

    summary_cols = [
        "dice", "iou", "surface_hausdorff_mm", "surface_hausdorff95_mm",
        "hausdorff_mm", "hausdorff95_mm", "pred_volume_ml", "volume_difference_ml"
    ]
    existing = [col for col in summary_cols if col in all_df.columns]
    if existing:
        summary = (
            all_df
            .groupby(["model", "checkpoint", "variant"], dropna=False)[existing]
            .agg(["count", "mean", "median", "std"])
        )
        summary.columns = ["_".join(col).strip("_") for col in summary.columns.values]
        summary = summary.reset_index()
        summary.to_csv(OUTPUT_ROOT / "summary_by_model_checkpoint_variant.csv", index=False)


print("Funzioni di inferenza pronte. Fix applicato: path assoluti + cache NIfTI pulita per MONAI LoadImaged + GT registration_mask.nii.gz.")


Funzioni di inferenza pronte. Fix applicato: path assoluti + cache NIfTI pulita per MONAI LoadImaged + GT registration_mask.nii.gz.


In [8]:
all_rows: list[dict] = []
errors: list[dict] = []

if not volumes:
    raise RuntimeError(f"Nessun volume trovato in {INPUT_ROOT}. Controlla path e pattern.")

for model_name, checkpoint_dict in CHECKPOINTS.items():
    for checkpoint_name, checkpoint_path in checkpoint_dict.items():
        checkpoint_path = resolve_project_path(checkpoint_path)
        print("=" * 100)
        print(f"Loading {model_name} | {checkpoint_name} | {checkpoint_path}")
        model = load_model(model_name, checkpoint_path, cfg, device)

        for idx, volume in enumerate(volumes, start=1):
            print(f"[{idx:03d}/{len(volumes):03d}] {volume.patient_id} | {volume.variant} | {model_name}/{checkpoint_name}")
            try:
                row = run_single_volume_inference(model, volume, model_name, checkpoint_name)
                all_rows.append(row)
                print(
                    f"  status={row['status']} | "
                    f"Dice={row['dice'] if row['dice'] is not None else 'NA'} | "
                    f"IoU={row['iou'] if row['iou'] is not None else 'NA'} | "
                    f"Vol={row['pred_volume_ml']:.2f} ml"
                )
            except Exception as exc:
                error_row = {
                    "patient_id": volume.patient_id,
                    "variant": volume.variant,
                    "model": model_name,
                    "checkpoint": checkpoint_name,
                    "image_path": str(volume.image_path),
                    "label_path": str(volume.label_path) if volume.label_path else None,
                    "error_type": type(exc).__name__,
                    "error_message": str(exc),
                    "traceback": traceback.format_exc(),
                }
                errors.append(error_row)
                print(f"  ERRORE: {type(exc).__name__}: {exc}")

        del model
        if device.type == "cuda":
            torch.cuda.empty_cache()

write_patient_csvs(all_rows)
write_csv(
    OUTPUT_ROOT / "inference_errors.csv",
    errors,
    columns=["patient_id", "variant", "model", "checkpoint", "image_path", "label_path", "error_type", "error_message", "traceback"],
)

print("=" * 100)
print(f"Righe metriche salvate: {len(all_rows)}")
print(f"Errori salvati: {len(errors)}")
print(f"CSV globale: {OUTPUT_ROOT / 'all_tavi_metrics.csv'}")
print(f"CSV per paziente: {OUTPUT_ROOT / '<TAVI_ID>' / '<TAVI_ID>_metrics.csv'}")

Loading SwinUNETR | best_metric_model | /Users/ricca/Desktop/Health_Informatics_Internship_2026/notebooks/output/models/SwinUNETR/best_metric_model.pth
SwinUNETR kwargs: {'in_channels': 1, 'out_channels': 2, 'feature_size': 48, 'use_checkpoint': True, 'spatial_dims': 3}
Loaded ema_model_state_dict from best_metric_model.pth
[001/008] TAVI_100 | denoising_contrasto | SwinUNETR/best_metric_model
  status=skipped_existing | Dice=0.7457828383432075 | IoU=0.5946201831252613 | Vol=74.56 ml
[002/008] TAVI_100 | original | SwinUNETR/best_metric_model
  status=skipped_existing | Dice=0.8800497175444362 | IoU=0.7857935582773102 | Vol=101.23 ml
[003/008] TAVI_100 | solo_contrasto | SwinUNETR/best_metric_model
  status=skipped_existing | Dice=0.7529638626630938 | IoU=0.603802760897593 | Vol=79.25 ml
[004/008] TAVI_100 | solo_denoising | SwinUNETR/best_metric_model
  status=skipped_existing | Dice=0.8681564617551487 | IoU=0.767028685874196 | Vol=102.84 ml
[005/008] TAVI_35 | denoising_contrasto | S

In [9]:
metrics_path = OUTPUT_ROOT / "all_tavi_metrics.csv"
if metrics_path.exists() and metrics_path.stat().st_size > 0:
    metrics_df = pd.read_csv(metrics_path)
    sort_cols = [col for col in ["patient_id", "variant", "model", "checkpoint"] if col in metrics_df.columns]
    display(metrics_df.sort_values(sort_cols).head(50))
else:
    print(f"Nessun CSV metriche leggibile trovato: {metrics_path}")

summary_path = OUTPUT_ROOT / "summary_by_model_checkpoint_variant.csv"
if summary_path.exists() and summary_path.stat().st_size > 0:
    summary_df = pd.read_csv(summary_path)
    display(summary_df.sort_values(["model", "checkpoint", "variant"]))
else:
    print(f"Nessun summary CSV leggibile trovato: {summary_path}")

errors_path = OUTPUT_ROOT / "inference_errors.csv"
if errors_path.exists() and errors_path.stat().st_size > 0:
    errors_df = pd.read_csv(errors_path)
    if len(errors_df):
        display(errors_df)
    else:
        print("Nessun errore registrato.")
else:
    print("Nessun errore registrato.")


,patient_id,variant,model,checkpoint,status,image_path,label_path,mask_path,raw_mask_path,spacing_x_mm,...,surface_hausdorff95_mm,hausdorff_mm,hausdorff95_mm,iou,pred_volume_ml,ground_truth_volume_ml,volume_difference_ml,volume_ratio_pred_to_gt,pred_voxels,ground_truth_voxels
16,TAVI_100,denoising_contrasto,SegResNet,best_metric_model,skipped_existing,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...,notebooks/output/tavi_to_infer_segmentations/T...,notebooks/output/tavi_to_infer_segmentations/T...,0.390625,...,23.948167,35.210424,23.439838,0.473684,54.942627,103.544312,-48.601685,0.530619,120024,226196
24,TAVI_100,denoising_contrasto,SegResNet,latest_checkpoint,skipped_existing,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...,notebooks/output/tavi_to_infer_segmentations/T...,notebooks/output/tavi_to_infer_segmentations/T...,0.390625,...,23.948167,35.210424,23.439838,0.473684,54.942627,103.544312,-48.601685,0.530619,120024,226196
0,TAVI_100,denoising_contrasto,SwinUNETR,best_metric_model,skipped_existing,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...,notebooks/output/tavi_to_infer_segmentations/T...,notebooks/output/tavi_to_infer_segmentations/T...,0.390625,...,12.345490,25.417654,9.965041,0.594620,74.555511,103.544312,-28.988800,0.720035,162869,226196
8,TAVI_100,denoising_contrasto,SwinUNETR,latest_checkpoint,skipped_existing,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...,notebooks/output/tavi_to_infer_segmentations/T...,notebooks/output/tavi_to_infer_segmentations/T...,0.390625,...,12.345490,25.417654,9.965041,0.594620,74.555511,103.544312,-28.988800,0.720035,162869,226196
17,TAVI_100,original,SegResNet,best_metric_model,skipped_existing,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...,notebooks/output/tavi_to_infer_segmentations/T...,notebooks/output/tavi_to_infer_segmentations/T...,0.390625,...,3.000000,9.592070,1.562500,0.771968,94.898529,103.544312,-8.645782,0.916502,207309,226196
25,TAVI_100,original,SegResNet,latest_checkpoint,skipped_existing,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...,notebooks/output/tavi_to_infer_segmentations/T...,notebooks/output/tavi_to_infer_segmentations/T...,0.390625,...,3.000000,9.592070,1.562500,0.771968,94.898529,103.544312,-8.645782,0.916502,207309,226196
1,TAVI_100,original,SwinUNETR,best_metric_model,skipped_existing,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...,notebooks/output/tavi_to_infer_segmentations/T...,notebooks/output/tavi_to_infer_segmentations/T...,0.390625,...,3.000000,5.740758,1.235265,0.785794,101.225739,103.544312,-2.318573,0.977608,221131,226196
9,TAVI_100,original,SwinUNETR,latest_checkpoint,skipped_existing,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...,notebooks/output/tavi_to_infer_segmentations/T...,notebooks/output/tavi_to_infer_segmentations/T...,0.390625,...,3.000000,5.740758,1.235265,0.785794,101.225739,103.544312,-2.318573,0.977608,221131,226196
18,TAVI_100,solo_contrasto,SegResNet,best_metric_model,skipped_existing,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...,notebooks/output/tavi_to_infer_segmentations/T...,notebooks/output/tavi_to_infer_segmentations/T...,0.390625,...,24.601262,35.338033,24.534639,0.445867,51.678314,103.544312,-51.865997,0.499094,112893,226196
26,TAVI_100,solo_contrasto,SegResNet,latest_checkpoint,skipped_existing,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...,notebooks/output/tavi_to_infer_segmentations/T...,notebooks/output/tavi_to_infer_segmentations/T...,0.390625,...,24.601262,35.338033,24.534639,0.445867,51.678314,10

,model,checkpoint,variant,dice_count,dice_mean,dice_median,dice_std,iou_count,iou_mean,iou_median,...,hausdorff95_mm_median,hausdorff95_mm_std,pred_volume_ml_count,pred_volume_ml_mean,pred_volume_ml_median,pred_volume_ml_std,volume_difference_ml_count,volume_difference_ml_mean,volume_difference_ml_median,volume_difference_ml_std
0,SegResNet,best_metric_model,denoising_contrasto,2,0.652760,0.652760,0.014005,2,0.484597,0.484597,...,19.204764,5.989300,2,83.500677,83.500677,40.387181,2,-54.495582,-54.495582,8.335229
1,SegResNet,best_metric_model,original,2,0.862335,0.862335,0.012695,2,0.758096,0.758096,...,1.865234,0.428131,2,142.483683,142.483683,67.295570,2,4.487425,4.487425,18.573160
2,SegResNet,best_metric_model,solo_contrasto,2,0.637213,0.637213,0.028943,2,0.467911,0.467911,...,19.753442,6.761634,2,80.833130,80.833130,41.231136,2,-57.163128,-57.163128,7.491275
3,SegResNet,best_metric_model,solo_denoising,2,0.853576,0.853576,0.007643,2,0.744593,0.744593,...,2.152717,0.227564,2,141.819044,141.819044,68.806593,2,3.822786,3.822786,20.084183
4,SegResNet,latest_checkpoint,denoising_contrasto,2,0.652760,0.652760,0.014005,2,0.484597,0.484597,...,19.204764,5.989300,2,83.500677,83.500677,40.387181,2,-54.495582,-54.495582,8.335229
5,SegResNet,latest_checkpoint,original,2,0.862335,0.862335,0.012695,2,0.758096,0.758096,...,1.865234,0.428131,2,142.483683,142.483683,67.295570,2,4.487425,4.487425,18.573160
6,SegResNet,latest_checkpoint,solo_contrasto,2,0.637213,0.637213,0.028943,2,0.467911,0.467911,...,19.753442,6.761634,2,80.833130,80.833130,41.231136,2,-57.163128,-57.163128,7.491275
7,SegResNet,latest_checkpoint,solo_denoising,2,0.853576,0.853576,0.007643,2,0.744593,0.744593,...,2.152717,0.227564,2,141.819044,141.819044,68.806593,2,3.822786,3.822786,20.084183
8,SwinUNETR,best_metric_model,denoising_contrasto,2,0.722861,0.722861,0.032416,2,0.566505,0.566505,...,8.635934,1.879641,2,99.509278,99.509278,35.289954,2,-38.486981,-38.486981,13.432456
9,SwinUNETR,best_metric_model,original,2,0.869528,0.869528,0.014881,2,0.769325,0.769325,...,1.590538,0.502432,2,146.252033,146.252033,63.676796,2,8.255774,8.255774,14.954385


Nessun errore registrato.


## Output atteso

Per ogni paziente viene creato:

`notebooks/output/tavi_to_infer_segmentations/<TAVI_ID>/<TAVI_ID>_metrics.csv`

Le segmentazioni vengono salvate in:

`notebooks/output/tavi_to_infer_segmentations/<TAVI_ID>/<variant>/<model>/<checkpoint>/segmentation_model.nii.gz`

Questa è la maschera finale. Se `RUN_LCC_POSTPROCESSING=True`, è post-processata con Largest Connected Component.

Viene salvata anche la maschera prima del post-processing:

`notebooks/output/tavi_to_infer_segmentations/<TAVI_ID>/<variant>/<model>/<checkpoint>/segmentation_model_raw.nii.gz`

Il CSV globale è:

`notebooks/output/tavi_to_infer_segmentations/all_tavi_metrics.csv`

La summary aggregata è:

`notebooks/output/tavi_to_infer_segmentations/summary_by_model_checkpoint_variant.csv`

Nota pratica: se vuoi confrontare solo `best_metric_model` o solo `latest_checkpoint`, elimina/commenta l'altro checkpoint nel dizionario `CHECKPOINTS`. Non serve sacrificare la RAM al dio dei loop inutili.